# 00 · Setup & Connection Check

This notebook verifies that:

1. The **`SERVICEBUS_CONNECTION_STRING`** environment variable is set.
2. The Service Bus namespace deployed by `infra/main.bicep` is reachable.
3. The expected entities (`demo-queue`, `demo-sessions`, `demo-dlq`, `demo-topic`) exist.

> If you haven't deployed yet, run `infra/deploy.ps1` first (see the [README](../README.md)).


## 1. Restore the SDK

We pin a specific version so the notebooks are reproducible.

In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"
#r "nuget: Azure.Messaging.ServiceBus.Administration, 7.18.2"

## 2. Load the shared `Config` helper

It looks for a `.env` file (walking up from the current directory), falls back to real environment variables, and exposes constants for entity names.

In [ ]:
#!import ../src/shared/Config.cs
Console.WriteLine($".env file: {Config.DotEnvPath ?? "(none — using process env vars)"}");
Console.WriteLine($"Connection string length: {Config.ConnectionString.Length} chars (looks good).");

## 3. Use the Administration client to list entities

The `ServiceBusAdministrationClient` is the **management plane** — use it for create / list / delete operations. The data plane (`ServiceBusClient`) is used for sending and receiving messages.

In [ ]:
using Azure.Messaging.ServiceBus.Administration;

var admin = new ServiceBusAdministrationClient(Config.ConnectionString);

Console.WriteLine("Queues:");
await foreach (var q in admin.GetQueuesAsync())
    Console.WriteLine($"  - {q.Name}  (sessions={q.RequiresSession}, maxDeliveryCount={q.MaxDeliveryCount})");

Console.WriteLine("\nTopics:");
await foreach (var t in admin.GetTopicsAsync())
{
    Console.WriteLine($"  - {t.Name}");
    await foreach (var s in admin.GetSubscriptionsAsync(t.Name))
        Console.WriteLine($"      • {s.SubscriptionName}");
}

## Concepts cheat sheet

| Concept | What it is |
|---|---|
| **Namespace** | The top-level container. Maps to a DNS name `<name>.servicebus.windows.net`. |
| **Queue** | Point-to-point. Each message is consumed once. |
| **Topic + Subscription** | Pub/sub. A copy of every message goes to each subscription. |
| **Sender / Receiver** | Data-plane clients created from `ServiceBusClient`. |
| **PeekLock** | Default receive mode. Message is locked, you must `Complete` it. |
| **ReceiveAndDelete** | Message is deleted on receive — at-most-once. |

Next: [`01-queue-send-receive.ipynb`](01-queue-send-receive.ipynb)
